## Empirical Tests on Real Data

### Download 4 Real Timeseries from Yahoo
Download daily historical data for Apple Inc. (AAPL) stock, the S&P 500 Index, the USD/JPY FX spot rate, and the Gold commodity spot price from 2001-01-01 to 2021-01-01

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from metrics import real_data_loading, predictive_score_metrics
from timeseries import log_returns, download_data

In [ ]:
start_date = "2001-01-01"
end_date = "2021-01-01"

In [ ]:
 # Single stock example: Apple Inc. (AAPL)
single_stock = download_data("AAPL", start_date=start_date, end_date=end_date)

# Stock index example: S&P 500 Index (^GSPC)
stock_index = download_data("^GSPC", start_date=start_date, end_date=end_date)

# FX spot example: USD/JPY (JPY=X)
fx_spot = download_data("JPY=X", start_date=start_date, end_date=end_date)

# Commodity spot example: Gold (GC=F)
commodity_spot = download_data("GC=F", start_date=start_date, end_date=end_date)

In [ ]:
# Display the data
single_stock.head()

In [ ]:
stock_index.head()

In [ ]:
fx_spot.head()

In [ ]:
commodity_spot.head()

### Calculate Returns and Plot

In [ ]:
single_stock['Close']

In [ ]:
single_stock_returns = log_returns(single_stock['Close']).squeeze()

In [ ]:
single_stock_returns

In [ ]:
stock_index_returns = log_returns(stock_index['Close']).squeeze()

In [ ]:
fx_spot_returns = log_returns(fx_spot['Close']).squeeze()

In [ ]:
commodity_spot_returns = log_returns(commodity_spot['Close']).squeeze()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_stock_prices_returns(prices, returns, dates, filepath, name):
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=True)

    # Plot stock prices
    ax1.plot(dates, prices, color='black', linestyle='-')
    ax1.set_title(name +' Prices')
    ax1.set_ylabel('Price')

    # Plot stock returns
    ax2.plot(dates[1:], returns, color='black', linestyle='--')
    ax2.set_title(name + ' Returns')
    ax2.set_ylabel('Return')

    # Set x-axis properties
    ax2.xaxis.set_major_locator(mdates.YearLocator())
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45)

    # Limit the number of x-axis labels to one per year
    ax2.xaxis.set_major_locator(mdates.YearLocator(1))

    # Adjust the space between the subplots
    plt.subplots_adjust(hspace=0.5)

    # Save the plot as a PNG file at 300 DPI
    plt.savefig(filepath, dpi=300)

    # Show the plot
    plt.show()

In [ ]:
single_stock.index

In [ ]:
single_stock_path = 'AAPL.png'
plot_stock_prices_returns(single_stock['Close'], stock_index_returns, single_stock.index, single_stock_path, 'AAPL')

In [ ]:
stock_index_path = 'GSPC.png'
plot_stock_prices_returns(stock_index['Close'], single_stock_returns, stock_index.index, stock_index_path, 'GSPC')

In [ ]:
fx_spot_path = 'JPY=X.png'
plot_stock_prices_returns(fx_spot['Close'], fx_spot_returns, fx_spot.index, fx_spot_path, 'JPY=X')

In [ ]:
commodity_spot_path = 'GC=F.png'
plot_stock_prices_returns(commodity_spot['Close'], commodity_spot_returns, commodity_spot.index, commodity_spot_path, 'GC=F')

### Run Empirical Tests

In [ ]:
from empirical_tests import run_tests

In [ ]:
lst_returns = [single_stock_returns, stock_index_returns, fx_spot_returns, commodity_spot_returns]
lst_labels = ['Apple', 'S&P 500', 'USDJPY', 'Gold']

In [ ]:
df = run_tests(lst_returns, lst_labels)

In [ ]:
df = df.set_index('Label')
pd.set_option('display.precision', 2)

In [ ]:
df

In [ ]:
df.to_latex()

### Plot Autocorrelation Functions

In [ ]:
from empirical_tests import acf_plots_block

In [ ]:
acf_block = 'ACF_real.png'

In [ ]:
acf_plots_block(lst_returns, lst_labels, acf_block)

#### Predictive Scores

In [ ]:
import torch
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
ssr2 = real_data_loading(single_stock_returns, 32)
sir2 = real_data_loading(stock_index_returns, 32)
fxr2 = real_data_loading(fx_spot_returns, 32)
csr2 = real_data_loading(commodity_spot_returns, 32)

In [ ]:
pssr = predictive_score_metrics(ssr2, ssr2, device=device)
psir = predictive_score_metrics(sir2, sir2, device=device)
pfxr = predictive_score_metrics(fxr2, fxr2, device=device)
pcsr = predictive_score_metrics(csr2, csr2, device=device)

In [ ]:
data_lst = [[pssr, psir, pfxr, pcsr]]
index_value = 'Predictive Score Real'
df = pd.DataFrame(data=data_lst, columns=lst_labels, index=[index_value])
pd.set_option('display.precision', 2)

In [ ]:
df

In [ ]:
df.to_latex()